# Tugas Nomor 3: Python Data Analyst

Notebook ini berisi jawaban lengkap bagian **3.1 sampai 3.6**. Jalankan melalui menu **Runtime → Run all** di Google Colab.

## Persiapan Data

Notebook akan mencoba membaca file dari folder Colab terlebih dahulu. Jika file tidak tersedia, data akan diambil dari Google Sheets yang digunakan pada materi KarirNex.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", None)

nama_file = [
    "showroom_mobil_10cabang_10k (1).xlsx",
    "showroom_mobil_10cabang_10k.xlsx",
    "Day 2 - showroom_mobil_10cabang_clean.csv",
]

lokasi_file = []
for nama in nama_file:
    lokasi_file.extend([Path(nama), Path("/content") / nama])

data_path = next((path for path in lokasi_file if path.exists()), None)

if data_path is not None:
    if data_path.suffix.lower() == ".csv":
        df = pd.read_csv(data_path)
    else:
        df = pd.read_excel(data_path)
    print("Data dimuat dari file:", data_path.name)
else:
    url_data = "https://docs.google.com/spreadsheets/d/1GuFkH44urw4ACqt5gsWhi1Scei0pGS2L/export?format=csv"
    try:
        df = pd.read_csv(url_data)
        print("Data dimuat dari Google Sheets.")
    except Exception:
        from google.colab import files

        print("Silakan unggah file CSV atau XLSX dataset showroom mobil.")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        data_path = Path(uploaded_name)

        if data_path.suffix.lower() == ".csv":
            df = pd.read_csv(data_path)
        else:
            df = pd.read_excel(data_path)

print("Data berhasil dimuat.")

## 3.1 Soal 1: Mengenali Struktur Data

Tampilkan jumlah baris dan kolom, nama semua kolom, serta 10 baris pertama.

In [ ]:
# 1. Tampilkan jumlah baris dan jumlah kolom dari df
jumlah_baris, jumlah_kolom = df.shape
print("Jumlah baris :", jumlah_baris)
print("Jumlah kolom :", jumlah_kolom)

# 2. Tampilkan nama-nama semua kolom
print("\nNama-nama kolom:")
print(df.columns.tolist())

# 3. Tampilkan 10 baris pertama dari data
print("\n10 baris pertama:")
display(df.head(10))

## 3.2 Soal 2: Jumlah Transaksi per Kategori Mobil

Hitung jumlah transaksi setiap kategori, urutkan dari yang terbanyak, kemudian buat bar chart.

In [ ]:
# Hitung jumlah transaksi per kategori mobil dan urutkan dari yang paling banyak
transaksi_per_kategori = df["category"].value_counts()

print("Jumlah transaksi per kategori mobil:")
display(transaksi_per_kategori.rename("jumlah_transaksi").to_frame())

# Bonus: buat bar chart
plt.figure(figsize=(9, 5))
bars = plt.bar(
    transaksi_per_kategori.index,
    transaksi_per_kategori.values,
    color="#2F75B5"
)

plt.title("Jumlah Transaksi per Kategori Mobil")
plt.xlabel("Kategori Mobil")
plt.ylabel("Jumlah Transaksi")
plt.xticks(rotation=20)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

## 3.3 Soal 3: Cabang dengan Transaksi Terbanyak

Hitung jumlah baris transaksi per cabang, urutkan dari yang terbanyak, lalu tampilkan cabang peringkat pertama beserta angkanya.

In [ ]:
# Hitung jumlah transaksi per cabang dan urutkan dari yang paling banyak
transaksi_per_cabang = df["branch"].value_counts()

print("Jumlah transaksi per cabang:")
display(transaksi_per_cabang.rename("jumlah_transaksi").to_frame())

# Tampilkan cabang dengan jumlah transaksi terbanyak
cabang_terbanyak = transaksi_per_cabang.idxmax()
jumlah_terbanyak = transaksi_per_cabang.max()

print("Cabang dengan transaksi terbanyak:", cabang_terbanyak)
print("Jumlah transaksi:", jumlah_terbanyak)

## 3.4 Soal 4: Bersihkan dan Siapkan Data

Langkah dilakukan secara berurutan dan hasil akhirnya tetap disimpan ke variabel `df`.

In [ ]:
# 1. Cek missing value di setiap kolom
print("Missing value sebelum dibersihkan:")
display(df.isnull().sum().rename("jumlah_missing").to_frame())

# Isi trade_in yang kosong dengan 0
df["trade_in"] = df["trade_in"].fillna(0)


# 2. Cek dan hapus duplikat penuh maupun duplikat order_id
duplikat_penuh = df.duplicated().sum()
duplikat_order_id = df.duplicated(subset=["order_id"]).sum()

print("Jumlah duplikat penuh sebelum dihapus:", duplikat_penuh)
print("Jumlah duplikat order_id sebelum dihapus:", duplikat_order_id)

df = df.drop_duplicates()
df = df.drop_duplicates(subset=["order_id"], keep="first").reset_index(drop=True)


# 3. Standarisasi payment_type: hapus spasi dan ubah menjadi Title Case
df["payment_type"] = df["payment_type"].astype("string").str.strip().str.title()


# 4. Ubah sales_date menjadi tipe data datetime
df["sales_date"] = pd.to_datetime(df["sales_date"], errors="coerce")


# 5. Pemeriksaan akhir
print("\nMissing value setelah dibersihkan:")
display(df.isnull().sum().rename("jumlah_missing").to_frame())

print("Jumlah duplikat penuh setelah dibersihkan:", df.duplicated().sum())
print(
    "Jumlah duplikat order_id setelah dibersihkan:",
    df.duplicated(subset=["order_id"]).sum()
)

print("\nInformasi akhir DataFrame:")
df.info()

## 3.5 Soal 5: Kombinasi Cabang dan Kategori dengan Pendapatan Tertinggi

Gunakan hanya transaksi berstatus `completed`, kelompokkan berdasarkan kombinasi `branch` dan `category`, lalu tampilkan lima kombinasi dengan total pendapatan tertinggi.

In [ ]:
# 1. Filter transaksi dengan status completed
df_completed = df[
    df["status"].astype("string").str.strip().str.lower() == "completed"
].copy()

print("Jumlah transaksi completed:", len(df_completed))


# 2. Kelompokkan branch dan category, lalu jumlahkan total_sales
pendapatan_cabang_kategori = (
    df_completed
    .groupby(["branch", "category"], as_index=False)
    .agg(total_pendapatan=("total_sales", "sum"))
)


# 3. Urutkan dari yang terbesar dan ambil lima kombinasi teratas
top_5_kombinasi = (
    pendapatan_cabang_kategori
    .sort_values("total_pendapatan", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

top_5_kombinasi["kombinasi"] = (
    top_5_kombinasi["branch"] + " - " + top_5_kombinasi["category"]
)

print("5 kombinasi cabang dan kategori dengan pendapatan tertinggi:")
display(top_5_kombinasi)

## 3.6 Visualisasi dan Tantangan Tambahan

Buat bar chart horizontal untuk lima kombinasi teratas, kemudian hitung rata-rata `price` per unit untuk kombinasi yang sama.

In [ ]:
# 4. Bar chart horizontal untuk 5 kombinasi teratas
data_chart = top_5_kombinasi.sort_values("total_pendapatan", ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(
    data_chart["kombinasi"],
    data_chart["total_pendapatan"],
    color="#2F75B5"
)

plt.title("5 Kombinasi Cabang dan Kategori dengan Pendapatan Tertinggi")
plt.xlabel("Total Pendapatan")
plt.ylabel("Cabang - Kategori")

plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda nilai, posisi: f"Rp{nilai / 1_000_000_000:.0f} M")
)

for bar in bars:
    nilai = bar.get_width()
    plt.text(
        nilai,
        bar.get_y() + bar.get_height() / 2,
        f" Rp{nilai / 1_000_000_000:.2f} M",
        va="center"
    )

plt.tight_layout()
plt.show()


# Tantangan tambahan: rata-rata harga per unit untuk 5 kombinasi teratas
rata_harga = (
    df_completed
    .groupby(["branch", "category"], as_index=False)
    .agg(rata_rata_harga_per_unit=("price", "mean"))
)

top_5_lengkap = (
    top_5_kombinasi
    .merge(rata_harga, on=["branch", "category"], how="left")
    [[
        "branch",
        "category",
        "total_pendapatan",
        "rata_rata_harga_per_unit"
    ]]
)

top_5_lengkap["total_pendapatan_rupiah"] = (
    top_5_lengkap["total_pendapatan"]
    .map(lambda nilai: f"Rp {nilai:,.0f}".replace(",", "."))
)

top_5_lengkap["rata_rata_harga_rupiah"] = (
    top_5_lengkap["rata_rata_harga_per_unit"]
    .map(lambda nilai: f"Rp {nilai:,.0f}".replace(",", "."))
)

print("Hasil akhir beserta rata-rata harga per unit:")
display(top_5_lengkap)

## Kesimpulan

Setelah seluruh cell dijalankan, notebook akan menampilkan struktur data, transaksi per kategori, cabang dengan transaksi terbanyak, hasil pembersihan data, lima kombinasi cabang–kategori dengan pendapatan tertinggi, visualisasi, dan rata-rata harga per unit.